In [1]:
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.common.by import By 
NOMBRE_GRUPO = "Agrotech"
CATEGORIA_ACTUAL = "Clima" # Los alumnos pueden cambiar esto segun su asignacion

try:
    client = MongoClient("database", 27017)
    db = client["proyecto_bigdata"]
    coleccion = db["datos_libros"]
    print("CONEXION EXITOSA: Listo para insertar datos de",
    CATEGORIA_ACTUAL)
except Exception as e:
    print("ERROR DE CONEXION:", e)

# 2. Ejemplo de logica de captura (Simplificada para la clase)
# Supongamos que ’lista_libros’ es el resultado de tu busqueda con Selenium
lista_libros = [
    {"titulo": "It’s Only Himalaya", "precio": " 45 .17"},
    {"titulo": "Full Moon over N o a h s Ark", "precio": " 49 .43"}
]

for libro in lista_libros:
    try:
        # Enriquecimiento del diccionario
        registro = {
            "item": libro["titulo"],
            "valor": float(libro["precio"].replace(" ", "").replace(",", "").strip()),
            "categoria": CATEGORIA_ACTUAL,
            "grupo": NOMBRE_GRUPO,
            "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S")
        }

        coleccion.insert_one(registro)
    except Exception as e:
        print("Error procesando libro:", e)

print("PROCESO FINALIZADO: Datos enviados a MongoDB.")


CONEXION EXITOSA: Listo para insertar datos de Clima
PROCESO FINALIZADO: Datos enviados a MongoDB.


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Cargar datos
spark = SparkSession.builder \
    .appName("Analisis_Categorias") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0") \
    .getOrCreate()

df = spark.read.format("mongodb").load()

In [3]:
df.printSchema()

root
 |-- _id: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- item: string (nullable = true)
 |-- valor: double (nullable = true)



In [4]:
df.show(5)

+--------------------+---------+-------------------+--------+--------------------+-----+
|                 _id|categoria|      fecha_captura|   grupo|                item|valor|
+--------------------+---------+-------------------+--------+--------------------+-----+
|69d64c23e2128cc0e...|    Clima|2026-04-08 12:37:55|Agrotech|  It’s Only Himalaya|45.17|
|69d64c23e2128cc0e...|    Clima|2026-04-08 12:37:55|Agrotech|Full Moon over N ...|49.43|
|69d6501ce2128cc0e...|    Clima|2026-04-08 12:54:52|Agrotech|  It’s Only Himalaya|45.17|
|69d6501ce2128cc0e...|    Clima|2026-04-08 12:54:52|Agrotech|Full Moon over N ...|49.43|
|69d6502ce2128cc0e...|    Clima|2026-04-08 12:55:08|Agrotech|  It’s Only Himalaya|45.17|
+--------------------+---------+-------------------+--------+--------------------+-----+
only showing top 5 rows



In [6]:
from pyspark.sql.functions import col

# Solo queremos registros que tengan un item real y un valor mayor a 0
df_filtrado = df.filter((col("item").isNotNull()) & (col("valor") > 0))

print("PASO 2: Limpieza completada.")
print("Registros originales:", df.count())
print("Registros validos:", df_filtrado.count())

PASO 2: Limpieza completada.
Registros originales: 22
Registros validos: 22


In [7]:
df.select("item", "valor").show()

+--------------------+-----+
|                item|valor|
+--------------------+-----+
|  It’s Only Himalaya|45.17|
|Full Moon over N ...|49.43|
|  It’s Only Himalaya|45.17|
|Full Moon over N ...|49.43|
|  It’s Only Himalaya|45.17|
|Full Moon over N ...|49.43|
|  It’s Only Himalaya|45.17|
|Full Moon over N ...|49.43|
|  It’s Only Himalaya|45.17|
|Full Moon over N ...|49.43|
|  It’s Only Himalaya|45.17|
|Full Moon over N ...|49.43|
|  It’s Only Himalaya|45.17|
|Full Moon over N ...|49.43|
|  It’s Only Himalaya|45.17|
|Full Moon over N ...|49.43|
|  It’s Only Himalaya|45.17|
|Full Moon over N ...|49.43|
|  It’s Only Himalaya|45.17|
|Full Moon over N ...|49.43|
+--------------------+-----+
only showing top 20 rows



In [8]:
df.filter(df["valor"] > 100).show()

+---+---------+-------------+-----+----+-----+
|_id|categoria|fecha_captura|grupo|item|valor|
+---+---------+-------------+-----+----+-----+
+---+---------+-------------+-----+----+-----+



In [9]:
df.groupBy("grupo").count().show()

+--------+-----+
|   grupo|count|
+--------+-----+
|Agrotech|   22|
+--------+-----+



In [11]:
# 2.Transformacion y Agregacion por CATEGORIA
# Esto permite comparar que generos son mas caros en promedio
reporte_categorias = df.groupBy("categoria").agg(
    F.count("item").alias("cantidad_libros"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("ANALISIS DE MERCADO POR CATEGORIA:")
reporte_categorias.show()

ANALISIS DE MERCADO POR CATEGORIA:
+---------+---------------+-----------------+-------------+-------------+
|categoria|cantidad_libros|  precio_promedio|precio_minimo|precio_maximo|
+---------+---------------+-----------------+-------------+-------------+
|    Clima|             22|47.29999999999998|        45.17|        49.43|
+---------+---------------+-----------------+-------------+-------------+



In [12]:
from pyspark.sql import functions as F

# 1. Ordenar por valor de forma descendente
# 2. Tomar el primer registro
ganador = df.orderBy(F.desc("valor")).limit(1)

ganador.select("item", "valor", "grupo", "fecha_captura").show()

+--------------------+-----+--------+-------------------+
|                item|valor|   grupo|      fecha_captura|
+--------------------+-----+--------+-------------------+
|Full Moon over N ...|49.43|Agrotech|2026-04-08 12:37:55|
+--------------------+-----+--------+-------------------+



In [14]:
from pyspark.sql import functions as F

# CONSULTA DE SALIDA: Resumen de integridad por Categor a
ticket_salida = df.filter(F.col("grupo") == NOMBRE_GRUPO) \
    .groupBy("categoria") \
    .agg(
        F.count("item").alias("total_libros"),
        F.round(F.avg("valor"), 2).alias("precio_medio"),
        F.max("fecha_captura").alias("ultima_sincronizacion")
    )

print(f"--- TICKET DE SALIDA: {NOMBRE_GRUPO} ---")
ticket_salida.show()

--- TICKET DE SALIDA: Agrotech ---
+---------+------------+------------+---------------------+
|categoria|total_libros|precio_medio|ultima_sincronizacion|
+---------+------------+------------+---------------------+
|    Clima|          22|        47.3|  2026-04-08 14:24:22|
+---------+------------+------------+---------------------+

